# Práctica: clasificando reseñas de compradores

En `reseñas.csv` tienes 20 reseñas de un e-commerce, cada una con su etiqueta de sentimiento (`positiva` o `negativa`). Es un dataset chico a propósito: no se trata de lograr el mejor accuracy posible, sino de que construyas el pipeline completo con tus propias manos y entiendas qué hace cada paso.
 
Antes de escribir código, leé las 20 reseñas del CSV a mano. Fijate qué palabras aparecen repetidas en las positivas y cuáles en las negativas. Esa intuición te va a servir después para saber si el modelo está aprendiendo algo razonable o si está fallando por algo que vos ya podías anticipar.

## Ejercicio 1: cargar y explorar
 
Cargá el CSV con Pandas. Contá cuántas reseñas hay de cada clase (`positiva` / `negativa`). Esto no es un paso decorativo: si las clases estuvieran muy desbalanceadas (por ejemplo 18 positivas y 2 negativas), un modelo podría lograr un accuracy alto simplemente prediciendo siempre "positiva", sin haber aprendido nada útil. Antes de entrenar cualquier modelo de clasificación, siempre chequeá el balance de clases.

In [39]:
import pandas as pd
db_path = "reseñas.csv"
df_reseñas = pd.read_csv(db_path)

print(conexion.head())

                                               texto sentimiento
0  el producto llegó en perfecto estado y antes d...    positiva
1  pésima experiencia, el paquete llegó abierto y...    negativa
2  muy buena relación precio calidad, lo volvería...    positiva
3  el vendedor nunca respondió mis mensajes, mala...    negativa
4         superó mis expectativas, calidad excelente    positiva


## Ejercicio 2: tokenizar sin librerías
 
Escribí una función `tokenizar_simple(texto)` que reciba un string y devuelva una lista de palabras en minúscula, sin signos de puntuación, usando solo métodos nativos de Python (sin NLTK ni spaCy). Pista: puede que necesites la librería `string` y su lista de puntuación, además de `.split()`.
 
Después, aplicá `word_tokenize` de NLTK sobre la misma reseña y compará los resultados. ¿En qué casos tu función simple se equivoca o produce un resultado distinto al de NLTK? Escribí al menos un ejemplo concreto de una reseña del dataset donde la diferencia se note.

In [40]:
import string

def tokenizar_simple(texto):
    texto = texto.lower() #pone en minusculas las palabras
    not_included_sings = "¿¡'" #esto es importante ya que esos dos signos no estan en el string.punctuation
    all_signs = string.punctuation + not_included_sings #aca uno las dos fuentes de datos en una
    texto = "".join(punt for punt in texto if punt not in all_signs) #for para quitar atenuaciones
    texto = texto.split()
    string_text = texto
    return(string_text)

texto = input("Ingrese el texto a transformar: ")
token_words = tokenizar_simple(texto)
print(token_words)

"""-----------------------------------------------------------------------------------------------------------------"""
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
tokens = word_tokenize(texto, language='spanish')
print(tokens)


['hola']
['hola']


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Ejercicio 3: el efecto de las stopwords
 
Tomá la reseña `"no lo recomiendo, una pérdida de dinero total"` (o equivalente del dataset) y sacale las stopwords en español con NLTK. Mirá el resultado.
 
Ahora respondé sin correr más código, solo pensando: si esta reseña fuera parte de un modelo de Bag of Words que sacó stopwords, y la palabra "no" desapareció, ¿qué información se perdió? Buscá en el dataset si hay alguna otra reseña donde sacar "no" cambiaría el sentido de la frase. Este ejercicio no tiene una única respuesta "correcta" en código: el objetivo es que argumentes cuándo sacar stopwords ayuda y cuándo perjudica.

In [41]:
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('spanish')) #esta es la lista de todas las palabras tipo stop en español

text = "no lo recomiendo, una pérdida de dinero total" #texto

#tokenizar para que el stop_words pueda funcionar
tokens = word_tokenize(text, language="spanish")

#filtrado de los stopwords
tokens_filtrados = [t for t in tokens if t.lower() not in stop_words]
print("Tokens filtrados:", tokens_filtrados)


Tokens filtrados: ['recomiendo', ',', 'pérdida', 'dinero', 'total']


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Ejercicio 4: vectorizar con TF-IDF
 
Usando `TfidfVectorizer` de sklearn, vectorizá las 20 reseñas del dataset completo (no hace falta separar en train/test todavía). Imprimí el vocabulario completo que generó el vectorizador con `.get_feature_names_out()`.
 
Buscá en ese vocabulario las 5 palabras con el IDF más alto (es decir, las más "raras" o distintivas del corpus) y las 5 con el IDF más bajo. Podés acceder a esos valores con el atributo `.idf_` del vectorizador, en el mismo orden que el vocabulario. ¿Las palabras con IDF alto te parecen informativas sobre el sentimiento de la reseña donde aparecen?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

"""primero pasar el dataframe a un estilo de lista para poder ser leido"""
reviews_list = conexion["texto"].tolist()
print(reviews_list)

"""ahora con tfifd muestor esa lista como una bag of words"""
tfidf = TfidfVectorizer()
matriz_tfidf = tfidf.fit_transform(reviews_list)

print(tfidf.get_feature_names_out())
print(matriz_tfidf.toarray().round(2))


['el producto llegó en perfecto estado y antes de lo esperado', 'pésima experiencia, el paquete llegó abierto y faltaban piezas', 'muy buena relación precio calidad, lo volvería a comprar', 'el vendedor nunca respondió mis mensajes, mala atención', 'superó mis expectativas, calidad excelente', 'se rompió a la semana de uso, no lo recomiendo', 'envío rapidísimo, tal cual la descripción', 'la caja llegó totalmente destruida y el producto dañado', 'funciona perfecto, instalación sencilla y buen material', 'tardó un mes en llegar y encima vino incompleto', 'excelente atención del vendedor, resolvió todas mis dudas', 'la calidad es muy inferior a lo que mostraban las fotos', 'quedé conforme, cumplió con lo prometido', 'no funciona como se anuncia, una estafa total', 'buenísima compra, ya la recomendé a mi familia', 'el color no coincide para nada con la foto publicada', 'llegó bien empaquetado y funciona de diez', 'cancelaron mi pedido sin avisarme, terrible servicio', 'precio justo y produ